# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # <--- this is a Metadata object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets by their @id
print("Available record sets in this dataset (by @id):\n")
for rs in metadata.record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', '<no name>')}")

# For each record set, list its fields (by @id and dataType)
for rs in metadata.record_sets:
    print(f"\nRecordSet: {rs.get('name', '<no name>')} (@id: {rs['@id']}) fields:")
    for f in rs['fields']:
        field_str = f"  - @id: {f['@id']}"
        if 'dataType' in f:
            field_str += f" (dataType: {f['dataType']})"
        field_str += f"; name: {f.get('name', '<no name>')}"        print(field_str)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare extraction of each record set (by @id)
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'RecordSet {record_set_id} loaded. Columns: {df.columns.tolist()}')
    else:
        print(f'RecordSet {record_set_id} contains no records.')

# For demonstration, pick the first available record set with data
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f'\nPreview of the first records from RecordSet: {main_rs_id}')
    display(dataframes[main_rs_id].head())
else:
    print('No record sets with data available.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Select numeric field and group field for EDA, using @id references
import numpy as np

# ---
# Tip: Review the output from the previous cells to pick the correct field @id. This code assumes you have them.
# Update these with actual field IDs as seen above.
# For demonstration, we'll heuristically select numeric and group fields:

if dataframes:
    df = dataframes[main_rs_id]
    # Try to auto-pick a numeric field and a group field by inspecting columns
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[col]):
            if numeric_field_id is None:
                numeric_field_id = col
        else:
            if group_field_id is None:
                group_field_id = col

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].dropna().mean() if df[numeric_field_id].dropna().mean() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (using field @id):")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a group field, if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (using field @id):")
            display(grouped_df.head())
        else:
            print("No suitable group field was found for grouping.")
    else:
        print("No numeric field found in the main record set for EDA.")
else:
    print("No data loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: visualize the distribution of the numeric field
import matplotlib.pyplot as plt

if dataframes and numeric_field_id:
    plt.figure(figsize=(8,5))
    values = df[numeric_field_id].dropna()
    plt.hist(values, bins=20, alpha=0.7, color='teal')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot versus group field if available and categorical
    if group_field_id and pd.api.types.is_object_dtype(df[group_field_id]):
        # Show mean per group
        plt.figure(figsize=(10,5))
        means = df.groupby(group_field_id)[numeric_field_id].mean()
        means.plot(kind='bar', color='purple')
        plt.title(f'Mean {numeric_field_id} per {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

_This notebook demonstrated how to connect, explore, and process the FAIR^2 Croissant dataset using field and record set `@id` references consistently. Update field IDs in EDA as needed for richer analysis. For details see the [original dataset metadata](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)._